# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

## Step 1: Parse the citations into `(cited, citing)` pairs

Drop the header line (it starts with `"CITING"`), split each line on commas, and convert to `int`. We key by `CITED` because the first join looks up the cited patent's state.

*While developing, you can add `.sample(False, 0.05)` here to work with a subset. The output below is for the full dataset.*

In [6]:
citations = rddCitations.filter(lambda line: not line.startswith('"CITING"')) \
    .map(lambda line: line.split(',')) \
    .map(lambda f: (int(f[1]), int(f[0])))   # (cited, citing)
citations.take(5)

[(956203, 3858241),
 (1324234, 3858241),
 (3398406, 3858241),
 (3557384, 3858241),
 (3634889, 3858241)]

## Step 2: Parse the patents

We need two things from the patent file:
* `patentLines`: `(patent, full original line)`, used at the end to print the augmented rows.
* `patentStates`: `(patent, state)`, only for patents that have a state. `POSTATE` is column 5 and is quoted (e.g. `"NY"`), so we strip the quotes. Foreign patents have an empty state and are dropped.

In [7]:
patentLines = rddPatents.filter(lambda line: not line.startswith('"PATENT"')) \
    .map(lambda line: (int(line.split(',')[0]), line))

patentStates = patentLines.map(lambda kv: (kv[0], kv[1].split(',')[5].strip('"'))) \
    .filter(lambda kv: kv[1] != '') \
    .cache()
patentStates.take(5)

[(3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA'),
 (3070806, 'PA')]

## Step 3: Look up the state of the *cited* patent

`join` matches on the key (the cited patent number) and gives `(cited, (citing, cited_state))`. We then re-key by the citing patent so the next join can look up its state: `(citing, (cited, cited_state))`.

In [8]:
citedStates = citations.join(patentStates) \
    .map(lambda kv: (kv[1][0], (kv[0], kv[1][1])))   # (citing, (cited, cited_state))
citedStates.take(5)

[(3858432, (3549216, 'PA')),
 (4061279, (3549216, 'PA')),
 (4674952, (3549216, 'PA')),
 (4767265, (3549216, 'PA')),
 (4806075, (3549216, 'PA'))]

## Step 4: Look up the state of the *citing* patent

Join again on the citing patent number. The result is `(citing, ((cited, cited_state), citing_state))`, the RDD version of the `Cited | Cited_State | Citing | Citing_State` table from the README. This join is slow, so we cache it.

In [9]:
bothStates = citedStates.join(patentStates).cache()
bothStates.take(5)

[(3858480, ((3435722, 'IL'), 'CA')),
 (3858582, ((3659602, 'PA'), 'CA')),
 (3858582, ((3542023, 'CA'), 'CA')),
 (3858582, ((3254449, 'CA'), 'CA')),
 (3858648, ((3714984, 'TX'), 'TX'))]

## Step 5: Keep same-state citations and count them per citing patent

Keep rows where `cited_state == citing_state`, emit `(citing, 1)` for each, and add them up with `reduceByKey`.

In [10]:
sameStateCounts = bothStates.filter(lambda kv: kv[1][0][1] == kv[1][1]) \
    .map(lambda kv: (kv[0], 1)) \
    .reduceByKey(operator.add)
sameStateCounts.take(5)

[(3890559, 2), (3891165, 2), (3923679, 1), (3930774, 2), (3954525, 2)]

## Step 6: Build the augmented patent lines

Left-outer-join the counts onto every patent line. Patents with no same-state citations get `None`, which we turn into 0. Then append the count as a new last column, matching the format in the README (e.g. `6009554,...,,6`).

In [11]:
augmented = patentLines.leftOuterJoin(sameStateCounts) \
    .map(lambda kv: (kv[0], kv[1][0] + ',' + str(kv[1][1] or 0), kv[1][1] or 0))   # (patent, augmented line, count)

In [12]:
# sanity check on the README example, patent 6009554. The real data gives 8, not 6:
# 8 of its 9 cited patents are from NY (the README's list of states is only illustrative)
augmented.filter(lambda t: t[0] == 6009554).map(lambda t: t[1]).collect()

['6009554,1999,14606,1997,"US","NY",219390,2,,714,2,22,9,0,1,,,,12.7778,0.1111,0.1111,,,8']

## Step 7: Show the top 10 patents by same-state citations

`takeOrdered` with a negated key returns the largest counts first, without sorting the whole RDD.

In [13]:
top10 = augmented.takeOrdered(10, key=lambda t: (-t[2], -t[0]))
for patent, line, cnt in top10:
    print(line)

5959466,1999,14515,1997,"US","CA",5310,2,,326,4,46,159,0,1,,0.6186,,4.8868,0.0455,0.044,,,125
5983822,1999,14564,1998,"US","TX",569900,2,,114,5,55,200,0,0.995,,0.7201,,12.45,0,0,,,103
6008204,1999,14606,1998,"US","CA",749584,2,,514,3,31,121,0,1,,0.7415,,5,0.0085,0.0083,,,100
5952345,1999,14501,1997,"US","CA",749584,2,,514,3,31,118,0,1,,0.7442,,5.1102,0,0,,,98
5998655,1999,14585,1998,"US","CA",,1,,560,1,14,114,0,1,,0.7387,,5.1667,,,,,96
5958954,1999,14515,1997,"US","CA",749584,2,,514,3,31,116,0,1,,0.7397,,5.181,0,0,,,96
5936426,1999,14466,1997,"US","CA",5310,2,,326,4,46,178,0,1,,0.58,,11.2303,0.0765,0.073,,,94
5980517,1999,14557,1998,"US","CA",733846,2,,606,3,32,241,0,1,,0.7394,,8.3776,0,0,,,90
5978329,1999,14550,1995,"US","CA",148925,2,,369,2,24,145,0,1,,0.5449,,12.9241,0.4196,0.4138,,,90
5951547,1999,14501,1997,"US","CA",733846,2,,606,3,32,242,0,1,,0.7382,,8.3471,0,0,,,90
